# **Build a Dataset Class for Horse Breeds**

https://www.kaggle.com/datasets/olgabelitskaya/horse-breeds

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("olgabelitskaya/horse-breeds")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'horse-breeds' dataset.
Path to dataset files: /kaggle/input/horse-breeds


In [ ]:
import os
import tarfile
import matplotlib.pyplot as plt
import numpy as np
import requests
import scipy
from PIL import Image
from torch.utils.data import Dataset, Subset, random_split, DataLoader
from torchvision import transforms
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
import torch

### Split the data into train val, and test set (starified)

In [ ]:
def stratified_splits(root_dir, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15, seed=42):

    image_paths = []
    labels = []

   
    for root, dirs, files in os.walk(root_dir):
        for filename in files:
            
            if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
                
               
                parts = filename.split('_')
                
                
                if len(parts) > 1:
                    label = parts[0]
                    
                    
                    full_path = os.path.join(root, filename)
                    image_paths.append(full_path)
                    labels.append(label)

    print(f"Total images found: {len(image_paths)}")

    
    X_train, X_temp, y_train, y_temp = train_test_split(
        image_paths, 
        labels, 
        test_size=(1 - train_ratio), 
        stratify=labels, 
        random_state=seed
    )

    
    relative_test_size = test_ratio / (val_ratio + test_ratio)

    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, 
        y_temp, 
        test_size=relative_test_size, 
        stratify=y_temp, 
        random_state=seed
    )

    print(f"Data Split Summary:")
    print(f"Train: {len(X_train)} images")
    print(f"Val:   {len(X_val)} images")
    print(f"Test:  {len(X_test)} images")

    return X_train, X_val, X_test, y_train, y_val, y_test

### Dataset Class

In [ ]:
class HorseDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
        
        # This maps "01" -> 0, "02" -> 1, ..., "07" -> 6
        self.breed_to_idx = {f"{i:02d}": i-1 for i in range(1, 8)}

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label_str = self.labels[idx]
        
        
        image = Image.open(img_path).convert("RGB") 
        
        if self.transform:
            image = self.transform(image)
            
        # Map the string label (e.g., "01") to an integer (0)
        label = self.breed_to_idx[label_str]
        
        return image, torch.tensor(label, dtype=torch.long)

### Transforms

In [ ]:
mean = [0.485, 0.456, 0.406]

std = [0.229, 0.224, 0.225]

transform = transforms.Compose([
    
    transforms.Resize((256, 256)),  
    transforms.CenterCrop(224),  
   
    transforms.ToTensor(),  
    
    transforms.Normalize(mean=mean, std=std),
])

In [ ]:
train_dataset, val_dataset, test_dataset = split_horse_data(transform)

print(f"Length of training dataset:   {len(train_dataset)}")
print(f"Length of validation dataset: {len(val_dataset)}")
print(f"Length of test dataset:       {len(test_dataset)}")

### Create Dataloader objects

In [ ]:
batch_size = 32


train_dataloader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)


val_dataloader = DataLoader(dataset=val_dataset, batch_size=batch_size, shuffle=False)


test_dataloader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)


train_features_batch, train_labels_batch = next(iter(train_dataloader))

print(f"Feature batch shape: {train_features_batch.size()}") # Expect: [32, 3, 224, 224]
print(f"Labels batch shape: {train_labels_batch.size()}")   # Expect: [32]

# Display the first label of the batch to see the integer mapping
print(f"Example Label (Integer): {train_labels_batch[0].item()}")

#### Display some images

In [ ]:
def show_batch(dataloader, breed_map):
   
    images, labels = next(iter(dataloader))
    
    
    plt.figure(figsize=(15, 10))
    
   
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    
    
    for i in range(8):
        plt.subplot(2, 4, i + 1)
        
        
        img = images[i].numpy().transpose((1, 2, 0))
        
      
        img = std * img + mean
        img = np.clip(img, 0, 1) 
        
        
        label_idx = labels[i].item()
        breed_code = f"{label_idx + 1:02d}"
        breed_name = breed_map.get(breed_code, "Unknown")
        
        plt.imshow(img)
        plt.title(f"Label: {breed_name}")
        plt.axis('off')


breeds_dict = {
    "01": "Akhal-Teke", "02": "Appaloosa", "03": "Orlov Trotter",
    "04": "Vladimir Heavy Draft", "05": "Percheron", "06": "Arabian", "07": "Friesian"
}


show_batch(train_dataloader, breeds_dict)
plt.show()

### Define Model 

### define Loss and Optimizer

#### Build one_epoch_training function loop 

#### Build one_epoch_validation function loop 

### Combine all to train the model
it should Save the best model and track train and val loss and accuracy


### test the model on test set

### show some predictions with the images

### Analyze the results
Is the model overfitting/underfitting?
Plot the training and validation loss/accuracy curves

### Load the model